In [4]:
import pandas as pd

In [5]:
import os
print(os.getcwd())

/Users/argira/Documents/ISYE7406/ISYE7406-Project


In [6]:
outcome = pd.read_csv("cleaned_outcome.csv")

In [7]:
intake = pd.read_csv("cleaned_intake.csv")

In [8]:
intake.columns

Index(['Animal ID', 'Name', 'DateTime', 'MonthYear', 'Found Location',
       'Intake Type', 'Intake Condition', 'Animal Type', 'Sex upon Intake',
       'Age upon Intake', 'Breed', 'Color', 'DateTime_intake'],
      dtype='str')

In [9]:
outcome.columns

Index(['Animal ID', 'Date of Birth', 'Name', 'DateTime', 'MonthYear',
       'Outcome Type', 'Outcome Subtype', 'Animal Type', 'Sex upon Outcome',
       'Age upon Outcome', 'Breed', 'Color', 'Age in Days upon Outcome',
       'IsEuthanized'],
      dtype='str')

In [10]:
print("Intake records:", len(intake))
print("Outcome records:", len(outcome))

Intake records: 173812
Outcome records: 173710


In [11]:
print("Repeated intake IDs:", intake['Animal ID'].duplicated().sum())

Repeated intake IDs: 17525


In [12]:
print("Repeated outcome IDs:", outcome['Animal ID'].duplicated().sum())

Repeated outcome IDs: 17519


In [13]:
outcome['IsEuthanized'].value_counts()

IsEuthanized
0    162886
1     10824
Name: count, dtype: int64

In [14]:
outcome.columns

Index(['Animal ID', 'Date of Birth', 'Name', 'DateTime', 'MonthYear',
       'Outcome Type', 'Outcome Subtype', 'Animal Type', 'Sex upon Outcome',
       'Age upon Outcome', 'Breed', 'Color', 'Age in Days upon Outcome',
       'IsEuthanized'],
      dtype='str')

In [15]:
euthanasia = outcome[outcome["IsEuthanized"] == 1]

In [16]:
print(euthanasia['Outcome Subtype'].value_counts(normalize=True, dropna=False).mul(100).round(2))

Outcome Subtype
Rabies Risk            44.93
Suffering              38.34
Aggressive              5.66
Medical                 3.28
NaN                     2.64
At Vet                  2.23
Behavior                1.63
Court/Investigation     0.95
Underage                0.34
Name: proportion, dtype: float64


In [17]:
pd.crosstab(
    euthanasia["Animal Type"],
    euthanasia["Outcome Subtype"]
)

Outcome Subtype,Aggressive,At Vet,Behavior,Court/Investigation,Medical,Rabies Risk,Suffering,Underage
Animal Type,,,,,,,,
Bird,0,2,0,0,12,0,136,1
Cat,4,124,0,0,101,111,2200,1
Dog,608,90,175,103,87,106,1032,0
Livestock,0,0,1,0,0,0,1,0
Other,1,25,0,0,155,4646,781,35


In [18]:
rates = (
    outcome.groupby("Animal Type")["IsEuthanized"]
    .agg(
        Total="count",
        Euthanized="sum",
        Rate="mean"
    )
)

rates["Rate"] = (rates["Rate"] * 100).round(2)

print(rates)

             Total  Euthanized   Rate
Animal Type                          
Bird           876         155  17.69
Cat          69385        2559   3.69
Dog          94466        2214   2.34
Livestock       34           2   5.88
Other         8949        5894  65.86


In [19]:
pets = outcome[
    outcome["Animal Type"].isin(["Dog", "Cat"])
].copy()

pd.crosstab(
    pets.loc[pets["IsEuthanized"] == 1, "Animal Type"],
    pets.loc[pets["IsEuthanized"] == 1, "Outcome Subtype"],
    normalize="index",
    dropna=False
).mul(100).round(2)

Outcome Subtype,Aggressive,At Vet,Behavior,Court/Investigation,Medical,Rabies Risk,Suffering,Underage,NaN
Animal Type,,,,,,,,,
Cat,0.16,4.85,0.0,0.00,3.95,4.34,85.97,0.04,0.70
Dog,27.46,4.07,7.9,4.65,3.93,4.79,46.61,0.00,0.59


Prepare the datasets to join intake and outcome

In [38]:
intake_df = intake.copy()

In [39]:
intake_df.columns

Index(['Animal ID', 'Name', 'DateTime', 'MonthYear', 'Found Location',
       'Intake Type', 'Intake Condition', 'Animal Type', 'Sex upon Intake',
       'Age upon Intake', 'Breed', 'Color', 'DateTime_intake'],
      dtype='str')

In [54]:
intake_df = intake_df.drop(columns=['DateTime_intake'])

In [40]:
outcome_df = outcome.copy()

In [41]:
outcome_df.columns

Index(['Animal ID', 'Date of Birth', 'Name', 'DateTime', 'MonthYear',
       'Outcome Type', 'Outcome Subtype', 'Animal Type', 'Sex upon Outcome',
       'Age upon Outcome', 'Breed', 'Color', 'Age in Days upon Outcome',
       'IsEuthanized'],
      dtype='str')

In [42]:
outcome_df['OutcomeDate'] = pd.to_datetime(outcome_df["DateTime"], errors = "coerce")

In [55]:
intake_df["IntakeDate"] = pd.to_datetime(intake_df["DateTime"], errors="coerce")

In [56]:
print("Missing intake dates:", intake_df["IntakeDate"].isna().sum())
print("Missing outcome dates:", outcome_df["OutcomeDate"].isna().sum())

Missing intake dates: 0
Missing outcome dates: 0


In [74]:
intake_df["IntakeDay"] = intake_df["IntakeDate"].dt.normalize()
outcome_df["OutcomeDay"] = outcome_df["OutcomeDate"].dt.normalize()

In [58]:
# Remove records missing matching information
import numpy as np

intake_df["IntakeRecordID"] = np.arange(len(intake_df))
outcome_df["OutcomeRecordID"] = np.arange(len(outcome_df))

intake_valid = intake_df.dropna(
    subset=["Animal ID", "IntakeDate"]
).copy()

outcome_valid = outcome_df.dropna(
    subset=["Animal ID", "OutcomeDate"]
).copy()

# merge_asof requires dates to be sorted
intake_sorted = intake_valid.sort_values("IntakeDate")
outcome_sorted = outcome_valid.sort_values("OutcomeDate")

# Match each outcome to the preceding intake
merged = pd.merge_asof(
    outcome_sorted,
    intake_sorted,
    by="Animal ID",
    left_on="OutcomeDate",
    right_on="IntakeDate",
    direction="backward",
    suffixes=("_outcome", "_intake")
)

print("Merged records:", len(merged))

Merged records: 173710


In [59]:
print("Matched outcomes:", merged["IntakeRecordID"].notna().sum())
print("Unmatched outcomes:", merged["IntakeRecordID"].isna().sum())

Matched outcomes: 152438
Unmatched outcomes: 21272


In [65]:
#check the unmmatched records to see if euthanized animals are disproportionately represented 
merged["MatchStatus"] = np.where(
    merged["IntakeRecordID"].notna(),
    "Matched",
    "Unmatched"
)

match_summary = (
    merged.groupby("MatchStatus")["IsEuthanized"]
    .agg(
        Total="count",
        Euthanized="sum",
        Rate="mean"
    )
)

match_summary["Rate"] = (
    match_summary["Rate"] * 100
).round(2)

print(match_summary)

              Total  Euthanized   Rate
MatchStatus                           
Matched      152438        5975   3.92
Unmatched     21272        4849  22.80


In [66]:
pd.crosstab(
    merged["Animal Type_outcome"],
    merged["MatchStatus"],
    margins=True
)

MatchStatus,Matched,Unmatched,All
Animal Type_outcome,,,
Bird,680,196,876
Cat,59519,9866,69385
Dog,86745,7721,94466
Livestock,29,5,34
Other,5465,3484,8949
All,152438,21272,173710


In [67]:
summary = (
    merged.groupby(
        ["Animal Type_outcome", "MatchStatus"]
    )["IsEuthanized"]
    .agg(
        Total="count",
        Euthanized="sum",
        Rate="mean"
    )
)

summary["Rate"] = (summary["Rate"] * 100).round(2)

print(summary)

                                 Total  Euthanized   Rate
Animal Type_outcome MatchStatus                          
Bird                Matched        680          45   6.62
                    Unmatched      196         110  56.12
Cat                 Matched      59519        1450   2.44
                    Unmatched     9866        1109  11.24
Dog                 Matched      86745        1643   1.89
                    Unmatched     7721         571   7.40
Livestock           Matched         29           0   0.00
                    Unmatched        5           2  40.00
Other               Matched       5465        2837  51.91
                    Unmatched     3484        3057  87.74


In [60]:
matched = merged.dropna(subset=["IntakeRecordID"]).copy()

In [63]:
matched.columns

Index(['Animal ID', 'Date of Birth', 'Name_outcome', 'DateTime_outcome',
       'MonthYear_outcome', 'Outcome Type', 'Outcome Subtype',
       'Animal Type_outcome', 'Sex upon Outcome', 'Age upon Outcome',
       'Breed_outcome', 'Color_outcome', 'Age in Days upon Outcome',
       'IsEuthanized', 'OutcomeDate', 'OutcomeRecordID', 'Name_intake',
       'DateTime_intake', 'MonthYear_intake', 'Found Location', 'Intake Type',
       'Intake Condition', 'Animal Type_intake', 'Sex upon Intake',
       'Age upon Intake', 'Breed_intake', 'Color_intake', 'IntakeDate',
       'IntakeRecordID'],
      dtype='str')

In [64]:
print("\nIntake records matched more than once:")
print(matched["IntakeRecordID"].duplicated().sum())


Intake records matched more than once:
1592


In [70]:
print("INTAKE DATASET")
print("Earliest intake:", intake_df["IntakeDate"].min())
print("Latest intake:", intake_df["IntakeDate"].max())
print("Total records:", len(intake_df))

print("\nOUTCOME DATASET")
print("Earliest outcome:", outcome_df["OutcomeDate"].min())
print("Latest outcome:", outcome_df["OutcomeDate"].max())
print("Total records:", len(outcome_df))

INTAKE DATASET
Earliest intake: 2013-10-01 07:51:00
Latest intake: 2025-05-04 23:44:00
Total records: 173812

OUTCOME DATASET
Earliest outcome: 2013-10-01 00:00:00
Latest outcome: 2025-05-05 00:00:00
Total records: 173710


In [71]:
unmatched = merged[
    merged["IntakeRecordID"].isna()
].copy()

intake_ids = set(intake_df["Animal ID"].dropna())

unmatched["IDExistsInIntake"] = (
    unmatched["Animal ID"].isin(intake_ids)
)

print(
    unmatched["IDExistsInIntake"].value_counts()
)

IDExistsInIntake
True     20457
False      815
Name: count, dtype: int64


In [72]:
# Keep unmatched outcomes whose IDs exist in intake
check = unmatched[
    unmatched["IDExistsInIntake"]
].copy()

# Find earliest available intake for each animal
first_intake = (
    intake_df.groupby("Animal ID")["IntakeDate"]
    .min()
    .rename("FirstIntakeDate")
)

# Attach earliest intake date to unmatched outcomes
check = check.join(first_intake, on="Animal ID")

# Calculate difference in days
check["DateDifference"] = (
    check["FirstIntakeDate"] - check["OutcomeDate"]
).dt.total_seconds() / 86400

print(check["DateDifference"].describe())

print("\nMissing intake dates:")
print(check["FirstIntakeDate"].isna().sum())

print("\nIntake dates after outcome:")
print((check["DateDifference"] > 0).sum())

print("\nWithin 1 day:")
print(check["DateDifference"].between(0, 1).sum())

print("\nWithin 7 days:")
print(check["DateDifference"].between(0, 7).sum())

count    20457.000000
mean         2.436039
std         46.512969
min          0.000694
25%          0.465278
50%          0.536806
75%          0.631944
max       2274.514583
Name: DateDifference, dtype: float64

Missing intake dates:
0

Intake dates after outcome:
20457

Within 1 day:
20347

Within 7 days:
20368


In [73]:
# Percentage of dates recorded at exactly midnight
for name, df, col in [
    ("Intake", intake_df, "IntakeDate"),
    ("Outcome", outcome_df, "OutcomeDate")
]:
    valid = df[col].dropna()
    midnight = (
        valid.dt.hour.eq(0) &
        valid.dt.minute.eq(0) &
        valid.dt.second.eq(0)
    )

    print(name)
    print("Records:", len(valid))
    print("Midnight timestamps:", round(midnight.mean() * 100, 2), "%")
    print()

Intake
Records: 173812
Midnight timestamps: 0.03 %

Outcome
Records: 173710
Midnight timestamps: 100.0 %



In [75]:
# Prepare records
intake_sorted = (
    intake_df
    .dropna(subset=["Animal ID", "IntakeDay"])
    .sort_values("IntakeDay")
)

outcome_sorted = (
    outcome_df
    .dropna(subset=["Animal ID", "OutcomeDay"])
    .sort_values("OutcomeDay")
)

# Match each outcome to the most recent intake
# on or before the same calendar date
merged = pd.merge_asof(
    outcome_sorted,
    intake_sorted,
    by="Animal ID",
    left_on="OutcomeDay",
    right_on="IntakeDay",
    direction="backward",
    suffixes=("_outcome", "_intake")
)

print("Merged records:", len(merged))
print("Matched outcomes:", merged["IntakeRecordID"].notna().sum())
print("Unmatched outcomes:", merged["IntakeRecordID"].isna().sum())

Merged records: 173710
Matched outcomes: 172785
Unmatched outcomes: 925


In [76]:
print("Match rate:",
      round(merged["IntakeRecordID"].notna().mean() * 100, 2), "%")

print("Duplicate intake matches:",
      merged["IntakeRecordID"].dropna().duplicated().sum())

print("\nEuthanasia outcomes by match status:")

merged["MatchStatus"] = np.where(
    merged["IntakeRecordID"].notna(),
    "Matched",
    "Unmatched"
)

print(
    merged.groupby("MatchStatus")["IsEuthanized"]
    .agg(["count", "sum", "mean"])
)

Match rate: 99.47 %
Duplicate intake matches: 315

Euthanasia outcomes by match status:
              count    sum      mean
MatchStatus                         
Matched      172785  10791  0.062453
Unmatched       925     33  0.035676


In [77]:
# Identify intake records matched more than once
duplicates = merged[
    merged["IntakeRecordID"].notna() &
    merged["IntakeRecordID"].duplicated(keep=False)
].copy()

# Sort for easier inspection
duplicates = duplicates.sort_values(
    ["Animal ID", "OutcomeDay"]
)

print("Outcome rows involved:", len(duplicates))
print("Unique intake records involved:",
      duplicates["IntakeRecordID"].nunique())

# Inspect examples
display(
    duplicates[
        [
            "Animal ID",
            "IntakeDay",
            "OutcomeDay",
            "Outcome Type",
            "IsEuthanized",
            "IntakeRecordID"
        ]
    ].head(20)
)

Outcome rows involved: 630
Unique intake records involved: 315


,Animal ID,IntakeDay,OutcomeDay,Outcome Type,IsEuthanized,IntakeRecordID
21531,A345209,2014-11-26,2014-11-26,Return to Owner,0,21409.0
21584,A345209,2014-11-26,2014-11-27,Return to Owner,0,21409.0
3351,A390222,2013-12-08,2013-12-08,Adoption,0,3278.0
3624,A390222,2013-12-08,2013-12-15,Adoption,0,3278.0
62595,A508053,2017-03-14,2017-03-14,Transfer,0,62251.0
63062,A508053,2017-03-14,2017-03-25,Return to Owner,0,62251.0
66504,A561806,2017-06-05,2017-06-05,Return to Owner,0,66707.0
66518,A561806,2017-06-05,2017-06-05,Return to Owner,0,66707.0
79147,A563384,2018-02-26,2018-02-26,Adoption,0,78861.0
81231,A563384,2018-02-26,2018-04-16,Adoption,0,78861.0


In [79]:
# Check repeated outcome information
duplicate_outcomes = duplicates.duplicated(
    subset=[
        "Animal ID",
        "OutcomeDay",
        "Outcome Type"
    ],
    keep=False
)

print("Rows with identical animal/date/outcome:",
      duplicate_outcomes.sum())

print("Rows with different outcome information:",
      (~duplicate_outcomes).sum())

Rows with identical animal/date/outcome: 104
Rows with different outcome information: 526


In [81]:
# Get original outcome columns
outcome_columns = outcome.columns.tolist()

# Find their corresponding names in the merged dataset
outcome_merge_columns = [
    col if col in duplicates.columns else col + "_outcome"
    for col in outcome_columns
]

# Check whether all columns were found
print(outcome_merge_columns)

['Animal ID', 'Date of Birth', 'Name_outcome', 'DateTime_outcome', 'MonthYear_outcome', 'Outcome Type', 'Outcome Subtype', 'Animal Type_outcome', 'Sex upon Outcome', 'Age upon Outcome', 'Breed_outcome', 'Color_outcome', 'Age in Days upon Outcome', 'IsEuthanized']


In [82]:
exact_duplicates = duplicates.duplicated(
    subset=outcome_merge_columns,
    keep=False
)

print("Exact duplicate outcome rows:",
      exact_duplicates.sum())

print("Other rows in duplicate-match groups:",
      (~exact_duplicates).sum())

Exact duplicate outcome rows: 94
Other rows in duplicate-match groups: 536


In [83]:
# Identify redundant copies, keeping the first
redundant = duplicates.duplicated(
    subset=outcome_merge_columns,
    keep="first"
)

print("Redundant records:", redundant.sum())

print("Redundant euthanasia records:",
      duplicates.loc[redundant, "IsEuthanized"].sum())

Redundant records: 47
Redundant euthanasia records: 1


In [84]:
euthanasia_groups = duplicates.groupby("IntakeRecordID")[
    "IsEuthanized"
].agg(["count", "sum"])

print("Intake groups containing euthanasia:",
      (euthanasia_groups["sum"] > 0).sum())

print("Groups with both euthanasia and other outcomes:",
      (
          (euthanasia_groups["sum"] > 0) &
          (euthanasia_groups["sum"] < euthanasia_groups["count"])
      ).sum())

Intake groups containing euthanasia: 8
Groups with both euthanasia and other outcomes: 3
